# MandiaCompact — étape 7 : reformulations françaises ancrées dans les sources

Calcul privé du projet https://github.com/mandi-model/mandia-compact ; fiche officielle de type modèle https://dataset.mandia.org/gnakalehacker/mandia-compact.

Deux bras de 400 mises à jour depuis le checkpoint CE sélectionné étape 6 : contexte/question canonique ou enrichissement français par l'assistant. 40 familles de réponses baoulé copiées de Wiktionary/Kaikki (CC-BY-SA-4.0), toutes issues du TRAIN de l'étape 5 ; 120 vues train, 40 validation, 40 test. Les références sont déjà connues : seuls la formulation française et le format question avec contexte changent. Ce n'est pas un test de nouvelles connaissances ou de conversation libre.

Les deux bras tirent les mêmes identifiants et les mêmes cibles. Leurs nombres de tokens source peuvent différer ; temps et tokens réels sont comptés séparément. Sélection gelée sur validation avec garde de régression avant tout test. 13 782 528 paramètres inchangés. AdamW/RNG/compteurs sauvegardés dans Drive tous les 100 pas. Les parents étapes 4–6 doivent être présents dans le même Drive. Aucune donnée baoulé inventée, aucun logits/poids privés de l'assistant, aucune validation native.

Méthode : augmentation de données française assistée par grand modèle, et non distillation de poids. Références : https://aclanthology.org/D16-1139/ et https://arxiv.org/abs/2305.02301. Les résultats de ces articles ne sont pas des résultats Mandia. Ce lot n'est pas un nouveau corpus de conversations réelles.


In [ ]:
# Mandia stage7: PRIVATE compute, public project is GitHub / model card.
from google.colab import drive
import pathlib,sys,base64,gzip,hashlib,json
if not pathlib.Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT7=pathlib.Path('/content/drive/MyDrive/Mandia/Sauvegarde-langues-MandiaCompact-2026-10-13/colab-runtime/mandia-compact-s2s-etape7')
ROOT7.mkdir(parents=True,exist_ok=True)
ASSETS7={'mandia_stage7.py': 'H4sIAAAAAAAC/51aa2/juJL97l/B2xcLSR1GsT22k7ZHwGZ7PIPgdqd7k+zcBbyGQEu0rY1eLcpJPEH++54iKdnOY3JngenYJotksZ6nivPhw4cbKaK1rI6FUomqZcx+rWQerZnYrDKZ16JOipyzZfKAqeI+Z72f/NOzrywrYpn6nX/kGDuuRbWSNSvXlVBJvmKqTJOas8tvN2yTKylbiqJiOXa8kywq8jtZKb29SJm8E+lG//A7Hz586CRZWVTYUdTrNFnw/1VgYi2U/lGJPC4yvop4VJTbhrQuqmjd/MhAkohQ1WIlB0wopgavTQ311PC1qZGeGnWWVZG1E/JHH/+YJf+qR6/71xz/Phf5Mll1rr59uwnUwKdPvxQQZX3imOXHUZGVIqqPVV8dy1qU8tTpXN9Mv1/zL1fBoNvlfXk87HRiuWTQAOTrVsW94nVxyyORF3kSiTT4VaRKeuMOq6TapHXwqMazOVtCsIolOZs5dSWS3OEOBJrEWqL4UUtVO/OnjiasiJC2xi6sxP3KOqhmTntGaMacOUuWrB1luAQDmdatMw8CexKTYIgmmlXYNKllFjw6qthUkXTGs8H8CLfw7a0MoecnsTqa9efgThuHM96jwX4iV/eycuZ7hGDaGcdJVLsVt4zbzXA1KExJqCWVuUsMzJrz597PQW/U1RfYTdpD9eSnPseBSayZN4Kd7a4690VZyjzWCz2SfL2pckvX6fy7tjw/yZeSPEeG5Bqup/X4Q4RpoZSrvYXTesVjeZdEWoXGh8j0XW9SFzXUCxHIXAVdoylYIi5E2hL5Srpd3rCvPN4b0RZM8AXf8pBsbiHqaG2mZ3rlWP896o3mzaFYcJ/Ua7iDDwes5UPteuM7ntPyPUb1pkSsmToKlmkhaveOmCT2joK8lcKjQ+tId6A8MfNaURjb49aKOzQEWtX4fDLWjkCQLLdhKZJKxqESGeSer1zisCpSXpsIRdcVgRH2bzKXlcB31/PhXRsYLcJM7A763mTxPo2RbrKTrHZDLc8HEkW0LgolWwZ6Iy68yXZvxrJEMwtvZ3pkNGSjc2NMO297gL+8ObmdH+7QWObL5S9naG2rCUhOhnEl7vMwl/chDgvhOiHsPjQrIfibaiO5k+Tlpg6hn1W9VmEmtmGcLGG/zXyjg7DI022I4I8TpJm0Osvg+65WiWHcyDzaxMJPcOSdSFKxSOEH2u0OZxfL3ihUm5KiKBQCARrztIozP1yHyB1vAqn/Mv394vM0MBMTQ6VgTfkmC+t1JUWs3IE30UE3u42TypUPSGZhcRsQyziAgmiRxqTCspIUmOnYskqKakjeM2wCz4xSDXyBtkw9LeiUBO2ChPY/cX5sijqJEyQ1u5NPS1LH84mR0DiVr2MHRCiV680phjfHjei40fvHjexxyBxpsdpI9ZePIycM3jzA7G4N+XhVFZs8lvHxD/H+/hCnWRY8y1PexLrM8wlu1WCNRaSpK3bmHASLZ7aNAERM/pGUjRM2iW3ehIN2wCPJ6kwbtEJDFLbrTkyQRWJqT4cNqLVwzRoPiUz0xVIOe/2zeLQQw+VZ91R246h/thDLZbwcdUen3bNFLD+NRr3uYPTp7KfT+FPcO4ukOIsWfRE7xMCWZGwtmL7a/XkmSuSASGfiwInKjcPvZbJa10o7V3AoGbsPLkcBMvlDViF47Q9HJCXLuIUXJzsarTKHdknTEElfBLOWCGxgVKbPzIe3BBhOkxfmtZun/U7uxYNIjxei2KTymGyiJXvPLfi7lszfM0UYnHwokSQIkBKuIIDmjE+5Y2SsM27aCGp8qF/+UpTjP5Ey1/dtKR9VXbklXCHVsBX5y92Ddp7XHmZcqySrbZTwxB1ygt1epZ8jQr+2JEW0cmEprgW8/nd88lmI0JvKMISNH2mS/WkKEQ2BZxnyV2mxcJ2Pfrl1PA8MANGXKoTsQlFlztjATSdFoP9yhVnkQ2c86HPnHtObEt+73BEwYaTtyMR8Z4NqgJJkA//56VmfD/tn5HS4Tg0U71BGSWroN/nDQM4x9k6xBVZZOP15yoiZHnAuWwI7rdl5LLJ/MvDGiDcSFjxWKC1n4yLOuMsbu0A6S4kdW55Y+IKszn7ApOhQdUIc6SJEKkbONUHBkawSKjD+Q5su7AgInBlgSD+Q5qSasLxoQhqDiSa1OrE+elLZCkUqzeEmpyQK0E3Y5s5oUPE70qHdwNeIxyXhW8yzFFlCHmakqyNpqFC60G3WMo2Pi03dVF13ibwH70t2ff51yv5x+e2fl8zWTs02nAyM2frl5ur84pL4r59VWruCCnxnyQPO8of7UvvPc4AX5vcZ0iK7/vrtC6NAsD3DWG+oB1FkIcDqgeY46CdXqd6WxruGUC8cONzq3OjfH+EItqtCyACO6DxblC1EKoCW48Px0eGKCYEbMNEb4BwbqCZ05ufpz4E1uSOwMbGrD4Z73QnQbg4UCIMr/pA5W0joSzJdDUEqhJcBOBebWJcfGj8xDX3AVhLDn7ATTIkQDOCKrO4SJQnMEHadMFgSCdIakxllBkfpJfBIlkmh4EYx1yoSzJ7JqBbc1JLt+xrYgwlkoroFc6ZMDvequLEu/bizQgoZ7wEqMjGDjEIKL25XhzxMIxIieBnKMKRqG9sgVqBWQggCMs6SKKQIa7HA9L+/T68uvk4vb0w+4bug672x5Hcs+PXi8/nNxbfLY5js9y8Xl7/Z1X+O6v88n/+dXckNysp6LYGacLlio9KtKRQSiE+78TFVYNCYXgeZU2VDriRjqC1foQpVjeKNjvwOmU6YVgQUvlxN9N/gyxXwTLUdv3/Bq+n3q4vrqb0fVpsrusTxkqJMil30nuYYwrXLxqClm/GFMLU7+xG0leHu/nu6bgu2SUFZ3zozbYGdX6ec7byPogtC93xS0kXpGEkHGfj51mpdxs0nMWHhWGaFOUuDt4ami9JFSz7UZmyrZVt84EL7xvqDOySE/aGCm8w9OhgtDaGOIaHeHSU+d2SKwL2giB8HQU/7k0viY4lil0UuqZXkFj8HNDZ7ftScYoJeUzYUL0+eU4TwPP5CbmN/9PFHI5Ajv/+xoD/lk1EnAYfQZFtd/9ja5nkVuuege1MhIV+aNj0sf1eQsixo+0lu209yP35ssUqkR8g/fKCQxj4yDTapZ1Wj9qPWSLtgN4ZFE6uorEPZVgWPthlEIYMT5CpN/8jd6wNZV3W4/eJxt83GS52vnLY4n5MoSD6qlJFyEy0am0ICmA1cZRuLLSWJYlkDbkh8XRWgxv5OinoNJ6I2Tv5tQGbV2NXMNbztwgWylj/kpg3G3cam7exMo9ktyM6432+ptNM0JIYnTdF7ttFwb6Mm1UmzWe/1zdrzBhwmZ2nmWp0HhjJRyQohHVE+sG1M38BCV9dn8SYrlfvo7CKuM959J0gG8EaqeuIKRXN4K7fKVA5NIQmrWMuHOFkht1Fp+1ak06HORMyYJvVX+LpWG29ilYHk2jZaxm0Y4F2fa1QZGDCpNRsMu3Tm60FwL3Vor3DbzU+ctjrjj42Bj19YPN835TG8ZmfsfxXeP+luDRxgRufPUUxoASBdOI1UniavZQLDriZB3qHOhU0Du8087QEpyyaryI+KlKKKexAJZFbW2zAij3F3NR9Byce72Y4TmNUObjQtM1MYa6RJR/qUEqgaf0IF2+uwaF0omQcZlEnTHBYSpCJbxEK7+HjH5uFBC9hLky/eyvLX0y/TzzoDnv9+fnlzfDO9biDCo2MQljM2DEBVtBdh5Nvx3dtH6cvc8t119lAzkQEthQAAse05eSb20pSb6dDStL+Rd5BsXiZT0/FuUpfJMQZ56gLxeUZ9k3wI8qFP7TR7Qhso3lpBFegrWfc5vbW23kAf8GbGfdp1pJ2V6WiaxL37oXRjgOQa7gZfkcVuY/5pRMb6d3ahQW60X0SZBrGBsZikyjtNoqRuagc1ZvV9wVQGT+fsdv/9B6dFt8qnBwAU8A/B48q0Oscrre4VKXuP7acJYUMVzCheEsED35oc1Ov1eX/wCcG21zvlvbMzk16YCPTGs2XbLHh8GHcH8dPxANBl8XJ2287q5bb71LxVIIk5c9Pacp3/yb9KpaA/NmaON+uZrtS/Qkhb65s07wSPuDU8YCbM9fnCfCKOJY28w0beTccVwBUFgWr7sNRrbhIx9cqs3KhciRK46t+CA4rFSwruLIp63UAqAh4HK1YvVwTBc4pKNo8ahmKnx5ngi7mnw2ljnjCL0BqJRv2IWYH+3MFEQ7pvu50XmdIUaUSjAuvwr0bkPXdQx+JO5E1EppXe5C9F4gYJNeFo/EoGj/abe6+msH+p1fcKYoueYzXiPTpIVK0TvysWw9S+bIj6/y8ak+yCt7GJ0ZcztnojrALj10mobQKFi+0B6G+TBb1YvMh1YVPnNc6x12ka9346PesP+2cAjUmWUDdm5gy6bRV+2CthIqUu9pZRa4SUa7S538dgbYqCE0JLTSfmvqhiqiKpinil2UQmgyqIUQOHwXl1D+mwBROlIqG+VvugjvIUpTBbbJl5bIfJUl9Az+i4AGEjjMegQcI5eCJXmtC+nR887VItRGXEK00sqpGsAU5QSqNywn/18/f+/Yd+xz4NPetmLGR9TwIkpU7e6GRQ136/mQFASXfda2lQq4hsUe2X9MUCMfmO2K/kCuGBehQ2kegWnn61ftX9K6EfkOxTujVwY626Hk8lUku567Hqvm1lG6SOR4/bJb1GEaV9pnLCsNxqyw9DR9+TTIZ6q7XpAtGQ6eICd8eKUqXr6HnH7GAm/xY41+f/9fv0t/OrX6bHpocwtX2D+Vsg660FwFiNPyCMhBn1IPE1rmAK7XvdO36mpQFPeXSog2waQy+72lQML7Y1UZYaZNPDDyAbEDSOeNZd9552olXmIU+f4mmMDdSDoOl8Pb/85eI8nN6cf5+ehjfTq68Xl1OHH9Q8zwHkYTx5hnocsayJ2KJLHRMPSd5Elk8cFg3LDIWKksT+jxR8mW7UunmEabMUGVEHBhLqlloYBgFMg8pG2MXYPH12/g8Zb7uSPCMAAA==', 'teacher-grounded-qa.jsonl': 'H4sIAAAAAAAC/+3d3Y7jxpUA4Pt9CsLAYG7S7ZaoGc8kF4txj7OwM5O14wwWARYwSlKpu9wUyeZPjzvBAn6IuQnmxneeNvYF9lpX/Rp5kj2nir9ilbrYLYot6gBJ7Ig/RYnn60MeFqv+8ZmYf/Z757OEs9k5j45OTk6eH5189jvnswVbCu8al70XF0IuwI/j0BOJ3CJiwsdPEhZf4AcRD2F9+GAeLHERfHQWBak/5/Mf5gz29cMlw8VhFCxDuYvTwE/4Twl3fu/8KfCZ7zAn9bkzO199uor48X/7b3kcszNc/l3KPYf5Ysm8bN0wiOPVpzk/So6E5/w77nnG/MAXM+b90F0bzI/f8wj3rFbxuDMNPHEsf5sgjWa80riuxcqKaeThWudJEsa///zzRXQMP3UiYKPo+jiIzj7HX/5z3H1lI48vlww3yz/3xIz7MZdf9vToy78dff/qaHJ88gcnjHjMoyvusCSJxDTFPf8OjoP5ZzyGLzp31C6dcxEnQXTteMK/iOW5j7g/O/8hiMSZOpEsjmEd5idHLE3Og4jPnZn6YVP4sb57JfcWsoiF5xGLYe/5mXeEn7fyR7nTPzh+kDi4F+4nYuZEfBZEuOJcMC84S7k8gCkLUo9XDoD/xGaJo/4/tPhfxe/kwJJlCGdhAb+6A8d4xp85f/3Lq6//7AS+ikefJeKK/xDxK8HfQzBOr39YwvEK/BUXzIs5/rizIJQ/4Tn35kdBmmSH68gvJPyzPzjfv3r7lfPXV3/5j6/+6vzx1duv33z91ff49SSEz6+YJ+YMD+rzhMeJ+popnBfuHyUsOuOJc8Z9HsFqf5erffY///YPHb5RX/jyYCeDZPDADY77MghHDSEThJFY3SRMRNyZ/+vnf9pgYSGc+kTACXVWv2R7I5AEchggXRuQZWtbU/mujFGdsDUpQBW+5RGc1suUr2fVWbBc8lwSySSZw5A5sUqV0MTWTJ4qZlnMX6bi7ivXb3n601GSQoitbmYCogyBFnhwZ5QzSeaQZLpjQxXHHXd4EfsGafoLOL0QqwsmljUWp5gBYVHM4T/wT69c20bdw3ZecpsytMbecz21ZjOtocG+dc6yj4nZoJiNemUGIaPCnbyRt4PwNu7BmwpMjHX4ZyIDXkQQKfzvlbiv8XgN8a8uMrGQk8hrTPJH/vbfn2vjr4NSTCVOVzewRgqBA7d8usQn77+mPA7g23uNXGn7qIL8kb9H6G9ilf+2WXCB9AchHsujQnIyF6onFasbDH6+MRXmxZf56mYW4SONrD5aXLpSWiSWe87yhana8qLDy9JvnjKBjxXCSMRVA69i5JYtsDLVYk8loD87iagVFMtdV/fXms3aXgs3+ecEZ1BwRj3Agb/tEKBTwf08So8JEAHaT0Dj3gDFTMRCXc4JuEpa3SSrGzgt9cu/r9QT8dxACOfWSQJBrIjVo2bl2rDqoM5x2uAk77XyYIf7ryXDU5xoUpYCCScB+5AFlLrI2OM2NrFKXdusZWSZKw/rgoouZWGRAgiwuFi9meqIGBF7vMSenRjqEs9OOrw6fHXNZMpKxFLMubY8h4txNRs9bXdXEsKPEuH41/ARW6S3H/WlvrUGWovC/etE5Z+TqEGJGvUnCrv/hTyKAx/+hXgRrwHyGvfDS3bsgIjJHivLYBVwkdd8sIxtw/0YnOwZLIdLME7WyNpeWnNtrHVQ48AwBTZL+CTSJrLGM2P40TNxn7KX4sgcmdtHcxOr/Lbl/htFOmvkrWZ+k10VeZbm5lp3YibIHtnbM3umThrPunxU5snOUUuh3u1e3SRRIBIThHJlG10P2XVFWiJ88dRHbQuhrzdqG2ptrb73glr2MUkblLTR7qW9qUVpyFPHwyM+JmyEbdjYxrvHdqqL0sa1JPzGkgNTFNPImaeEjtANAZ1rg66L92CKUGUyycl3YCQubabLKihMqGfNlPOI3yD4Taxy3pbLKM0shvG6YGLq3TXgyIyuPEnhsBQ+N/UueX7S6UB5i4VwfgzS/PXPs1TAMdf9vebq5Wvhx0mUSh3ldm1Gq9xiW9UBf3DxNHDOYG+mAX8MLbf2iG3oQOafk8hBiRz1JRJzBs++DsYsxm5ERIkoEV0jOu41acZw/Yr9yHROvltDIjdQ25JFsjhAi66NxS7e8skGsaxFq3zjoDRXY/N9GlG+JKOHaHRilS+3XOrJSeYDd1UozIOaB23hx8eaTKTdkpIqgR0y2JGpKjTq/AL3KhCyBBryKAoARLJhWOZ8/XaJ80EtrOt7n/oOuwg26Wu2134YootAOwyR+pjoDYreqC96DCJ09Sv5I38H7W+8e3+n6r5RRmpZit1kpLou0SN6w6Dn2tDrpkNOEaKS3lWKZ1RTwylfr8BRKJSV6sqEkTAOBOPEKg92MS1QNUizebjyAL6riw5fW588ksdBeDRN6fy8yxkt37JIsCWvzs7lLOE3CHwTjfUtjpIj2YfchuGWWytJ5qvh/Fwp05I0t91+jq5UWyjNPiaUg0I56g8lK2P0mDgSR+JomvW5U47v8shcm102C2HL2Z7zgCeYBHOIMF0bmB1UdvJ4Nc3Krk+gWNspNm096zP5JJ/75nNilTi7KPbkEbs2C7QOZjGklIh4bfrnYjtKo8R0sEy/mBhqQF9MOu8bMEu5wFD3mAM/z1nauK4t1mj7GORee14vtyYiEXLzjRXXZlP3GHo7MQy9ndDLkcPzNurLGz7RCLyEwBG4gwI37gschAucxCJQ5TtW8hl/NNW9YbWI0qL/TRblbV/gIIfk8NE6dG0cdtETpwAoZ0yScSu4sTOOuoc8yiZ1qWxyrx45JJJEPlqRE6vM2El3nMJk2bcmi987uuPInx8T5SVdr5LKwal8MTYUZF6MO57qgsn3/2dsynzeuDaFE8Pl+4O45n2e/G+hhfow4PDpj8I3Dv9dbas1uPqOC2/Zx8RtUNxG/XCrlOnzOD0mdsTuUNiNd8/uXRGaaz1lNBNelD7bzChD1Ija46Pm2lDraD4nUycZc86bgsqmz1aTyxBDYvj4GE6sMl4X1RaMUzVDtQGetsxSQJGbNkASRsK4rxifmYosz3ZQZMHYWIZyDrVA1zNMLX7oHd8DW2mygyh3FoJvtLfWaHuC0IjWYPY5IRwUwlE/CKs9JOsBe0wYCeOBYhzvHuO79Ri9uy7jOcl1KEceVFs+7LqUXJLLx+7StXG54+rNHWmzXjxVLzupLQgn4RwSzolV0uy6prOJo6a0AyRTL0lRc5pteplmc3gXckkqSR2K1NGJYagbWNDlveYFJj9/jr3VAB0OzpQ2O37jRHGJXNeK3H32WQEGH13A5kJ1RE8TQ981QzOtkWFbT07dJ69OddRw6ep/CdvgsI36wYZ5MZjN0nB1g3eDKoAjTvgI3wHhG/eHrzrTm4U+vAKEswLfRe6BzeBWEQ/Q9qUn8kgeH79H18ZjN+89ZaGq3mJSEaxefIKQ1mEsRiEGlFcigTSK9R5Kj8RxOBwnVumxk5LNhSrZ3GGkGJJmdTOLMIHKVxEzh+tZdiGrsqSTdA5A58jw8hMs6PjiFV/Nk6MrhbD5Ogm1sE0WbLO7OjHwdRGy1Dfaqu26vaq1fZeg8gXkaVCeRv14Wk9TMmqjPGyPSRgJG4qw8e6FvVGxKZVhtMJ9mv6mrjqWRXYBqaKbmBGz/WLm2jDrcEruLFDXIGmG4vaEeqEhX+WoVTmTwBG4xwFuYpXXtlknyUoHsewvprtUNEzOZCiNZNecJI/k7ZO8L0w1kC86vmfLJw/D78r1k4O2uUNrs7u6J5zozJ9vqC/Wd90aFOxbxyn7mDANCtOoH0y1ia9JFIkajqjx7kW9ySKzOpGu7hF2PqtCHtJt7rxIF+l6BLpcG13d1jkUteK+qjFH7oYJqymdEbg9AzexSmdb7Q+iRr0tIrzK7q53d0pxhc/yUpPkkby9kffCVOd40fWtGU6PPk8d7ykerJ9o5ulTK7RJZu13Woe19DB8xAUzylpvobUtbEKHK/+cdA1K16gvXaubMwYJK5vqBEP1mIgRsQESG++e2JsClZSGXfRVzFp3ACmBEjyCt5/wXBt4XRZJSoUVVzKGda/VzM6DWD09Zqo3iFyTE0ACuKcAJ1aZb9udQ3wOZyKBTSBcQi+NGw6Blo7h5n4jeeyTRtK4hxpdU4cRt+tn3Ksbb/UJYmXKWToLUjkWT4gnIYkfPMzCvfddt5YApovbD9g/y/fPpszcR2tDg/fpsdVordprq1xIEAcFcdQPRCkAcyEEMb9CGRC8frCcRpD48ig+JpNk8gBNjvtNjqkP5w7PpvJYCeHGnWKsuq8A2coO2r0+QESJ6P4RdW2IdlDOed3MknDmcChpzKOrm7jbsVIIK2HdP6wTq3za5fgpG8J480Aq9UtkhB7DJ3LEQE5XwiR3yHKfmcpEz7q8Ev4T/JJ+Ni3XlEXTem+zU3zTDxSWQLlzJeLsBlZtazep+1ZaqU7wjh/i6IQXzP87M83vvt5qa4pq9zqH5RJCOCiEo74QhkGUcIJIEAmihDjuLxvO5aC4TSD1/jq8WJ+luRNCSAiHhNC1QdjF2LjNoN+UH9XEYxHeyoXn17FAnHhHKuCHvUyxDx6cdcqTRHSARCdWeXKb9R4NTjkRGZy2q7SYP77pxtDNpw6ccilBHSLUl6byzstOL2gXCyGjhIUY/bE8ZcLQ10CtbcfunvutQsMPYQfO2ZRfeNwAzdBOa2+NRgpv5RLyNihvo768zVgoZ3Ugb+TtgLyNd+/tTRGhql+AitzVDVc9A2Q4b+jGUwt3+448pJE0PnqNro3GDl/BUsErZ8C9E0plnrEs5tvONEYkieSjJzmxSpDbrNSocI25EzMRQ+DP9TCahZlLTZ7ML2vJI3ncf4/PTVNOP3/e+RPGePUJjsLBs3fOpiIxsmj3vKL9btfrnu+924+ORAf/Aol8Q/Wz2Vhrcu+9J6cvn7z8UocOj4TIDY7cqC9yMvJjSG9wKASP4B0avHFf8JZwiHjqrla/JRyfxsU49XsmcZMTdsVnlSgnjsRxOBxdG44dVGe+Lz3Jgql0mL9IJQN6Y21GuqzWZuI0WtdNTsnpYJxOrNJmFy9TqajF+mnj0rXEZjEvNa9eAM85ZVSSOjypL04MxZwXJ10OI/CeYWE1cX4MUqA5T50p89giqOfQ11KhB185TqJUzT2YbXgks6va2urVxq03WHnfEVcRsGzpBY5x8njjAbQmKhvS+SwWENBBAR0RUAJKQB8v0HE/QPFid8ZCNvVkr+yNcr4jp+T00J26Nk47qB1hzIZBDLdnc/kmRcyuAhEdwZ1fxNULVnfaxbGWgfpMDsezTGMUyuWOceBluZz4Et9B851YpdltlpS+WQtSmXLx3SU17rkaaDmPaMn5jt5Bc70sbSYnz+R5wJ5HpsLTqOPxKzFS20jJNmk3XOXDG1kbUCtQEM3z0623SP7I32Z/ox794ePP1CeDZPCwDY77MYgXnDz7Mq1rRzLK2wwqSRAJ4mOH6NpA7PCtr8pNZjF/D0ZxDcv3aUS5kUgeCMmJVW7c8vg81UoPjqpcQSDHVda6zAs9Pg6bE2m3o6RJQgcm1DTP+YuuZ2K+CgQer8PEmdcYzk4tbSOtze7qpt7DVhe3Hzxx+9HMqr779oOayw55E0+YO+Zlh0DABgds1A8wFvJo9SspI2UHoWy8e2XvskiV15lXqXE6HjWSgFcIanOPR+JI3CMV59qI67DakodtWWvBaG7qK6bCUgpLqISQEO47wolV2tvqK1rqvaoi6qsU7+g5U1FYzm9eXKaSRtK41xpNgxy/6HIQyK+97GXm6qNAOJseesNflvmrX5l+1DkPoinN1vbxstTurcgumiyhfokdC5wL5jtT7/bDIjVZtT6K1oSn6v3JySI1G86PjRAPDvGoH8QZiTJwMbDhrC0DfDZAfIkv8bXhO34kORgnb+U/hZHAp4mwQO/qK18+BSxpVTYj02SaTKNp18Z0NxNxVdIx1pFYzkXewIpK3rbM1hBxHtwMZ9DgBMql8vE/QSfoBw59YpW8uxhxqJqhS9yGIL+rzlX8lZiDLfj3LMVX/1wQdsJ+yNhfmkagftnlqJzfspBlU9eBPb9ZPi5e8Q5xTRuk99lliXDKpgwVXYS3H3w4V3qCjTZaUwuzujS2oWMWXqj2idngmI16YlbEPWkjbQejbdyTNjkHbRSEkVjdJHJQknwqWp2T+kzuoQp48kf+9t6fa+Ovg1LRuzxaHRbCmUwE3uStfpEMmn2SgumP8I2aa5JBMrj/BidWObCLKo4MeDntnoGHvlOS4likwsIPaSSNe61xfGLolAQLOrwi9RhExerTFVyFLjE4sGLoPYUjmvJ1HWp5ZQMbcw/efwVg4AkIFWd2+wEIaPVtaK21w5limGnUSSwPhBwOyuFo9w7flJG7PlqIjGQ5mB0mQBnMx2STbB6mzfHubb6uyJM6MX5XN/LRWsVEc8yAQFTJxBy+dBpFosUIskSUiO4dUdeGaJeD+1S9XmpoaN779GQvvoa3I7JKVodsdWKVTrc66k8ZuHBSYxzDx3xt2ywCZQ9CGuFPQkno8ISOTUWhcac3o9mIWiJJIVBm2bAJ3E/XM6collj5u/d+S3dvbz8m0e3HWOC4WUJfg93QTGt4S9XfLYnUP2Ohs7fMD4r4DY7f6JHwY3DjeAU3nSSQBB6WwPHuBeZ3kt56HF9mMDZqzBYSSAI5SJCuDcgO6ztGlTo3snwT8igOsMdPsRbhJJyDxDmxypZbLejgWEFZPlSjLwe+eutIzbQlBWYVnmqY31HouaRsSmCHDtY1TAgPCzq8vL2AX9KvFCVxUrkUzthZsJwGuvypFv+p3Mx++oOttlUyVbudC+fH1D/DifySyDDW+uYDaD/qumpPO+56uYiMDsroaPdGFYANfYHyOGYpuSW35FbndtyXWxW22AtcBe3cACdbUW1ELInlQbB0bVh23EFoPX3KXgUVJs0+Qsl1KENebUdwCe7BwZ1Y5dMt9xZSYXq5lixlaUh36asd5Kfcx8braoJMkIcPeWIqOk1OOp7pSHb4g5/x1XXjTWv4kVpOcWS/s/rY8h5EU8rkZsah5av7bi1KNaADVS4hT4PyNOrHE6a4wF8EUUKmyNTATI17MHWthudRoZq/4qWZKazEEMI5ldsRMSK2b8RcG2IdFGLy2Yiqt2BZ7EY6GHgbBecEDkFNmDmbwU49NT8f9694FBNBIriXBCdWWa6LEXhQTDG5Vxa22krKHK4yIxynblaZwU86oeRH8vZU3shUAxl1eH35NxY4S/y/2XDj3JkyOKa0ZuE/V/9Xrmj/wP5B+y614dJF6iyX3BOOnxqGuTI21r7/Gzak7fOWLyB2g2I36ocdX05ZdJmSPJJ3qPLGPclbf9aGi6NNVl7jIjl8DmkkjQPV6Npo7LITTC1g5bPxIpprVlY/O1M4dUiXTJLJIZucWGXIbRZjymBd/ZIxS3U4mr1agoyLIb0SS2I5DJYTU6Vm0vHTdYbDN2G8wsmDEFmG4gw4xc0JOvIV2zxpv//O65XRJHD8ecTNZVFTQ63ZYTM6dfnnhG5Q6EaEjtARut2iG+8e3et6iKoJRPGQktVNbOj/Up2UKnsemEc8USSKg6Do2lDs9CWlust1aeZBjCsr2ZdoyCSZfPQmJ1bpceuD20BYJHKSRtXhE3tbx2m0kYlh4OJsH/WajWqC21ZtSCpJfeRSn5tKNs87ftaIM6/BGfQTOAt+M0NerX5LuMyRuHI+3Vu75xgPbaJeN8XB6XBfvheAVt+52FQ+rTfdfma5rCHtrHKVZURxUBRH/VCsTCpOIkkkiSxFjncv8l0ZpmuTGUMYbzLTWJlYEsthsnRtWHZQ8cmf4sPCMBKrm4ThyxZ4W2gGhLd2csOZnECnxEQ6SecgdU6skmYXL0phzOYzlevJaMacqZq8rM4tTimUkA4U6QtT2edFh1e237KwUaZM4ziF+DGXQ0PcyEbhA/deAvySTVW59WzKpqnhNcZNrbWvuKqWtEXXchEBHBTAUU8AjU80SCSJPGiR492LfF2L01qPHq669KCHDX16MjBt+vQQS2K5TyxdG5bdd+/Jja6b0/TuieA8evW17J+TEE/iuU88J1ZZs6OePlnkN3r66MAYZygPN14Wk1kyOyyzppnKJy87fmNEjYSNA22ns4gbZtho85ZIux3WO9TNBWx14W0Ypmpt761pyd3rYBULiNWgWI36YWUYIp+ckbNhOhvv3tmbLDolNgzY1Y39G1cqwAkbYdtDbK4Nti7nEM/l1Tltfqmq5RSKxI7YPTJ2E6sct82ySjEyL3ZT0zpo86pUq5mayB/5e0z+XprmXXrZ7YSki4UoIjV2piKepSKJjVOR4uo2vO6/45KZ+hCgwcbiwhN6asaWWoPLmtGRqywidINCN+oLnbGEQgpJ4aEpHO9e4ZtKlKpeMKrGovrAyHje2AmmGvFkkSwOxqJrY7HT6ksF5po0cwmmspJ9EYZMksm9MDmxyo/bLM2ogK0UZww6jF1cNl7lEk7COQicrmsYgRgW7OoWchZ4q09r9ctTVAtpMebwnyI92rO7367X4SXC8a+Z719Hwl+ktx838VtrsX2vsrwhbb+y6kLiNyh+o774QbRIAJDWPEYKSeEhKxz3pRAAJLAJxItBSjZhFNwr4ugZiQxxgkgQBwrRtYHYQfnmTZECZex6//r5n6ubZSDn5cb7wTxb6iqrcg7vYu1adm1XyiGn5HRPnE6sEub2SzpZwMIaQs4rZbh01c/vvcaYZJLMocl8aarndNnXG5KcnOEeX6ubw7n5EU4YvhF4njZTpZPwZRjL9TE51la2ArndxkqiYIL5TuB4tx8dNjfh3NT6fYwyk09GNodnc7R7m6+qIfq7uh7CSTgJZ45zvHuceNd5zjyeRtkoqasbvIbkG+Tkl5vyLpVJQ6ubiCVpxMkreT0gr66N125KRRio/upGjWRch4AuZTzr7xKLG1I4EcpTcxcEl+AOGe7EKtFus3b0tedcOyx/1JllXDMa7ZjHCpF8eSsPf2JLbA+E7XPXUFh67nZ4ffwNd0IerX4NROwsAzhhzJMTCCRBGsu+bktYUoOz+lk+LJU/++oG/tWfYdVX7gJJJ5W92IDdwRFU3sz0z7zAuXgPS5njp77jy3qxs4STyS7gGp89NQxrbneYrY2rRnXIyyWkfFDKR70ohziFQ6iFLgQKPrEh4SSchG9T+Hj3wt9WAnaZvShzxaN4dbOB2XepPOdZ0BNtok2076Dt2tDuoCT2TYHEWTIH5CxTX1alsUPUEiJMmsICWPMuNxuyKLpOzpc4Oqv8w6Cm/plzIk/kifwm8hOrbL7V0aTXiFbjN+ZFhUl78d6srK1+0fCT7PCOIHaSxt8T+jNAfwboz0Dtz4DpLb7nXb7A8JbBuYMzLMPjaZbGF0GQTJnn6bpI4ypJsZGV4620UUp9K0V6gSduPzrXT28/wBZT5t9+MPjc3Pw9XGLTT05fPnn5pV6nOjTyOTifo937fFOGajYIbx7MhJNwEs4S53j3OL/BgTVFnEcwXN3ipbQBT3XoJuHImVDUVpycktPDceraOO2gvFVGsHOXHmUHVoTAS7Kp7oWfwM3r6rekcMssZ7MnuSR3EHInVhm2ixnuKxErs+eGK2HDYFBqHKlEJCmnK2Tye3h+vzD1/fqiy2fGf2OBwxxoUf6MzJ/F6x2qp4JnMw/CN4GMKrewHz7x4Q2UMnEJ7OYiSIKL1Hfgv3qRmjZbM8waeTI+gf/VOawcBVEcHMVRPxTDII5Xn/BxD4kkkSSyInLcV3JMfe6oAU43ucwuLdWKVcoEk2AOGqZrA7ObNwSVNjlZdp44MaDxHuwuq9gTAOODsichPQCkE6vs2UVpqBq2kqqM5rveBMxtVxw1mBNX4jpErqZp3txO57rJE2ck4DBMVVN0KeB0yy6CCzgL/kzI4RxbktxOQ3WaiYAAvGb+lKcbxn2rt96+Olu2sKFEWz0M0jk4naP+dOLNaMijWHZLIKpElahupDruh2pxnYpez+AacJ5pjeM7R6zB/ujXEIMElaAeDlTXBmpHVaRNKORIU5IQ3iPCDde5vC9tQstWxvtGuZDcktsDcDuxSrDbLCxh1M5FDClWCphyls6CNHQqKTbePEw5gMrXpCRLWAeJ9f8BZ4Vs8jrHAgA=', 'AUDIT-DONNEES.json': 'H4sIAAAAAAAC/11Sy27jMAy85ysEn5P6Hcfd66I/USwEWqJtoYpk6OG0KPrvS8ttFtkLBUrD4XCozwNj2QhXpRX67Jk1xXG7ETaasOWflFEeHChDaVmld7pZQSsJQVlzr9pw6EPKKf1KTIQB7meo2jM9ZLUYsRZVh/KCVVUKkM2l6Np27BFLKEZBhzh37aUe20vXNEJCU3V9XWHf9GUxZIl0ABs1ctCa4zuIwL2NTiBPMnkAN2FSH1zEVGDwxr+L/r3uo0IMM5qgBHcorJMouVSg7RSTISNov3OMDo2Y+arw5vnwwcF75QOY8NCILFmRqDYYygcCcsER+7cADsJZ7xMf94tWj4qX6BbrcfNMWBOc1RolezP2Zk47A3tJgtgCDpbZgUfm7BB9MOj9L2ZsYMpIXJCCCYws+ClEWl7cd5d6WS35TMHGcFdncEW3G5rGuAtzOOLWOLnzuq99DmHxz3kOQpMfRGWnjyfrpvx3eT6VZd3n2fE/pHtXa4LA4POqLtqngmKZEexP6nNFIto606cCMaM77YaTC5Oj70mL+nEA4nSlEdNExzT4DdU0h5x0qMAkVSmt94EPX4e/Pt6qWvQCAAA=', 'CREDITS.txt': 'H4sIAAAAAAAC/zWPy07DUAxE93zFLEFKAgvYwKotQqp4LFoQYmkSN7FyH8G+Sdu/5zZVlx7PHI+/pU8SA+nx9pWk7wUWR60ZfCA/OLYCq1W5/Cm3i/K+uitgzBhYS437i/Vr8wYKzWXsxFLUY4UlxdEx6jgIN5hYfymJx06jhyVq+QGfm8X64+mSXD/bDHLsPUE5kQRuKrxnUQhkltEU0iNelEPdZXRIfEgjOfyNbKdPzoSBlIZOydhwXXcUWrabAiEmhFxiYkzkpKFT4iwr11Gb3DMzc1WbV1ZhS57Rh7gPSKQtJ+zIi5MMlgDO1iPmSxJa2OAkVVf//DIej1UBAAA='}
HASHES7={'mandia_stage7.py': '6044fb5633f0d0438000f005849f78db20c3b6c5edc2b693530818b53322ea41', 'teacher-grounded-qa.jsonl': '3cfe3c27ed8e221cad480755f9ee1a0fcee1c67583f58744cda427932e94910b', 'AUDIT-DONNEES.json': 'e20d55fd41faf3bcdfcf717de2b08813cce182b709623f02c50ed8ad2e4fe99f', 'CREDITS.txt': '270242f29e6f595a52dac0c7bbab715c8e7be232a9cf04ba93aa49a8f036c4f5'}
for name,encoded in ASSETS7.items():
 data=gzip.decompress(base64.b64decode(encoded));assert hashlib.sha256(data).hexdigest()==HASHES7[name]
 dest=ROOT7/name
 if dest.exists():assert hashlib.sha256(dest.read_bytes()).hexdigest()==HASHES7[name], 'Frozen source differs: use a new run directory'
 else:dest.write_bytes(data)
FROZEN_REPLAY={
 'mandia-compact-s2s-etape4/parallel-prepare.jsonl':'dc2bfd9ba783c5edec159ec35bfbb66206598520bef31f960da3ad92cc240cbd',
 'mandia-compact-s2s-etape4/replies-prepare.jsonl':'29334943cd507b8ffa9385b530b9625aec82c0741aeade3575a30c7df7e2caf1',
 'mandia-compact-s2s-etape4/data/waxal-baoule-text.jsonl':'4f7730d32d3324d2cc819f33d7633b2d59e01f81c8e2f38651d0377050ec69d4',
 'mandia-compact-s2s-etape5/quotidien-prepare.jsonl':'370d3bb631a4ad4765fbe3b15e0f6019c0470ce03c687f87ac71f6fda6b3048c',
 'mandia-compact-s2s-etape6/dialogues-prepare.jsonl':'9375c6a0918a5bf8056cecd42df0c8aff90c3439220bc45bfe10e8cc700a2184'}
for rel,expected in FROZEN_REPLAY.items():assert hashlib.sha256((ROOT7.parent/rel).read_bytes()).hexdigest()==expected, 'Changed parent data: '+rel
sys.path.insert(0,str(ROOT7.parent/'mandia-compact-s2s-etape6/code'))
sys.path.insert(0,str(ROOT7))
import mandia_stage7
report7=mandia_stage7.main()


MandiaCompact — étape 7 terminée le 14 octobre 2026

Deux continuations de 400 mises à jour sur L4, architecture propre inchangée (13 782 528 paramètres). Parent : CE étape 6. Contrôle : question/contexte canonique ; essai : reformulations françaises rédigées par assistant, cibles baoulé recopiées exactement de 40 exemples du TRAIN Wiktionary/Kaikki étape 5.

Les références sont connues dans tous les splits. Ce test réserve seulement des formulations françaises de questions avec contexte : aucune preuve de généralisation à de nouvelles réponses ou de conversation libre. Aucun nouveau dialogue naturel enregistré ajouté ; aucune validation native.

Sélection gelée avant test : teacher_french, pas 50 ; score validation 2,67705 contre 2,73884 pour le témoin, également pas 50. Les pas 100–400 ont été écartés par la garde sur les anciens dialogues. Les derniers checkpoints de reprise à 400 et les meilleurs poids à 50 sont conservés séparément.

| Test de reformulation, 40 vues / mêmes 40 familles connues | Parent | Canonique | Français enrichi |
|---|---:|---:|---:|
| chrF++ | 14,114 | 29,347 | 32,592 |
| Références exactement retrouvées | 0/40 | 4/40 | 6/40 |
| CE ↓ | 3,296 | 1,460 | 1,342 |

Les anciennes pertes test se dégradent légèrement : étape4 3,66286 → 3,68805, étape5 4,19511 → 4,24213, étape6 4,98540 → 5,05705. Les 14 anciennes réponses simples apprises restent exactes. Il s'agit d'un compromis, pas d'une amélioration générale établie.

Deux diagnostics avec question identique et contexte différent : le bras enrichi donne la même réponse pour « pain » et « sucre », et ne retrouve pas correctement les deux références pour « singe » et « aigle ». 0/2 paires entièrement exactes. Un changement de sortie seul ne constitue pas une compréhension.

Relecture technique des 40 sorties : noms de personnes et objets encore confondus, réponses génériques, faits omis. Exemple : contexte Mariame possède un mouton → « Konan le bua » ; contexte Aka mange du pain → « Aka di sukla. », identique à la sortie du contexte sucre. Ces erreurs sont établies par confrontation aux références sources ; la relecture n'est pas une validation native de grammaire ou de naturel.

Budget : mêmes identifiants tirés et 231 414 tokens supervisés par bras ; 732 256 tokens réels pour le contrôle, 747 192 pour le français enrichi (+2,04 %). Temps incluant validations et sauvegardes : 111,19 s / 110,17 s. Une seule graine ; pas de revendication de gain de vitesse ou de significativité.

Méthode : augmentation française assistée par grand modèle, pas distillation de poids/logits. L'assistant n'a pas produit de nouvelles cibles baoulé ni de raisonnement privé. Les articles de distillation motivent l'essai ; leurs gains ne sont pas ceux de Mandia.

Les 6 fichiers lourds totalisent 551 778 812 octets, tous sauvegardés et hachés sur Drive. Aucun fichier de poids enregistré sur la machine locale. Le code, les données préparées, audits et résultats rejoignent le dépôt GitHub public ; les poids d'inférence de cette étape portent une étiquette de recherche QA. Aucun Space créé et aucun backend public d'inférence connecté.
